# Capstone Topic 3: Gold Layer & Streaming

> **Phase 7 → Week 16 (Capstone) → Topic 3**

Build the Gold aggregation layer and a Structured Streaming pipeline for real-time order metrics — the final two tiers of the ShopStream Medallion architecture.

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import os, shutil, json, logging, time
from datetime import datetime, timezone

spark = SparkSession.builder \
    .appName("ShopStream Capstone - Gold & Streaming") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

BASE    = "/tmp/shopstream_capstone"
RUN_DATE = "2024-01-15"
RUN_ID   = f"run_{RUN_DATE.replace('-','')}_001"

# Ensure Gold and streaming dirs exist
for path in ["gold/revenue_by_region", "gold/revenue_by_tier",
             "streaming/live_orders", "checkpoints/live_orders"]:
    os.makedirs(f"{BASE}/{path}", exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("shopstream")

def info(msg, **kw):
    log.info(json.dumps({"ts": datetime.now(timezone.utc).isoformat(),
                         "run_id": RUN_ID, "msg": msg, **kw}))

print(f"ShopStream Capstone — Gold & Streaming | run_id={RUN_ID}")

## Part 1: Rebuild Silver (prerequisite)

Recreate the Silver data so this notebook is self-contained and runnable independently.

In [ ]:
# Recreate Silver so this notebook runs standalone
silver_data = [
    ("ORD-001", "USR-100", "laptop",     1299.99, "shipped",   "US",   "premium",  "2024-01-15 09:00:00", "2024-01-15"),
    ("ORD-002", "USR-101", "phone",       799.00, "pending",   "EU",   "standard", "2024-01-15 09:05:00", "2024-01-15"),
    ("ORD-003", "USR-102", "tablet",      499.50, "shipped",   "APAC", "standard", "2024-01-15 09:10:00", "2024-01-15"),
    ("ORD-004", "USR-100", "headphones",  149.99, "cancelled", "US",   "basic",    "2024-01-15 09:15:00", "2024-01-15"),
    ("ORD-005", "USR-103", "monitor",     329.00, "shipped",   "EU",   "standard", "2024-01-15 09:20:00", "2024-01-15"),
]
cols = ["order_id","user_id","product","amount","status","region","tier","event_time","date"]

silver_df = spark.createDataFrame(silver_data, cols) \
    .withColumn("event_ts", F.to_timestamp("event_time")) \
    .drop("event_time")

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
silver_df.write.format("delta").mode("overwrite") \
    .partitionBy("date").save(f"{BASE}/silver/orders")

print(f"Silver rebuilt: {silver_df.count()} rows")
silver_df.show(truncate=False)

## Part 2: Gold Layer — Revenue by Region

Gold tables are **aggregated, business-ready** data. Each Gold table answers one specific business question.

Gold rules:
- Read from Silver (never from Bronze)
- One write per business metric, partitioned by date
- Idempotent: same output for the same input, every time
- Always include `_updated_at` so freshness can be monitored

In [ ]:
def build_gold_revenue_by_region(silver_df: DataFrame, run_date: str) -> DataFrame:
    """Daily revenue, order count, and avg order value per region."""
    return silver_df \
        .filter(F.col("date") == run_date) \
        .groupBy("date", "region") \
        .agg(
            F.count("order_id").alias("order_count"),
            F.round(F.sum("amount"), 2).alias("total_revenue"),
            F.round(F.avg("amount"), 2).alias("avg_order_value"),
            F.countDistinct("user_id").alias("unique_customers"),
        ) \
        .withColumn("_updated_at", F.current_timestamp()) \
        .withColumn("_run_id", F.lit(RUN_ID))


def build_gold_revenue_by_tier(silver_df: DataFrame, run_date: str) -> DataFrame:
    """Daily revenue breakdown by order tier (premium / standard / basic)."""
    total_rev = silver_df.filter(F.col("date") == run_date) \
        .agg(F.sum("amount").alias("grand_total")).collect()[0][0] or 1.0

    return silver_df \
        .filter(F.col("date") == run_date) \
        .groupBy("date", "tier") \
        .agg(
            F.count("order_id").alias("order_count"),
            F.round(F.sum("amount"), 2).alias("total_revenue"),
        ) \
        .withColumn("revenue_pct",
            F.round(F.col("total_revenue") / F.lit(total_rev) * 100, 1)) \
        .withColumn("_updated_at", F.current_timestamp()) \
        .withColumn("_run_id", F.lit(RUN_ID))


def write_gold(df: DataFrame, path: str, run_date: str, name: str) -> int:
    """Idempotent Gold write — dynamic partition overwrite."""
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    df.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("date") \
        .save(path)
    count = spark.read.format("delta").load(path) \
                 .filter(F.col("date") == run_date).count()
    info(f"gold_{name}_written", rows=count, path=path)
    return count


# ── Execute ────────────────────────────────────────────────────
t0         = time.time()
silver_read = spark.read.format("delta").load(f"{BASE}/silver/orders")

gold_region = build_gold_revenue_by_region(silver_read, RUN_DATE)
gold_tier   = build_gold_revenue_by_tier(silver_read, RUN_DATE)

n_region = write_gold(gold_region, f"{BASE}/gold/revenue_by_region", RUN_DATE, "revenue_by_region")
n_tier   = write_gold(gold_tier,   f"{BASE}/gold/revenue_by_tier",   RUN_DATE, "revenue_by_tier")

info("gold_complete", duration_s=round(time.time()-t0, 2))

print(f"\nGold — Revenue by Region ({n_region} rows):")
spark.read.format("delta").load(f"{BASE}/gold/revenue_by_region").show(truncate=False)

print(f"\nGold — Revenue by Tier ({n_tier} rows):")
spark.read.format("delta").load(f"{BASE}/gold/revenue_by_tier").show(truncate=False)

## Part 3: Gold DQ — Freshness & Completeness

Gold DQ is simpler than Silver DQ because data is already clean. The key checks are:
- **Completeness**: did all expected regions appear?
- **Freshness**: was `_updated_at` set in the last 3 hours?
- **Sanity**: is total revenue positive and non-zero?

In [ ]:
EXPECTED_REGIONS = {"US", "EU", "APAC"}
FRESHNESS_HOURS  = 3

def run_gold_dq(df: DataFrame, run_date: str) -> dict:
    rows = df.filter(F.col("date") == run_date).collect()
    if not rows:
        raise ValueError(f"DQ FAIL: Gold has 0 rows for {run_date}")

    actual_regions = {r["region"] for r in rows}
    missing        = EXPECTED_REGIONS - actual_regions
    total_revenue  = sum(r["total_revenue"] for r in rows)

    # Freshness: compare _updated_at to current time
    latest_update  = df.filter(F.col("date") == run_date) \
                       .agg(F.max("_updated_at")).collect()[0][0]
    age_hours      = (datetime.now() - latest_update.replace(tzinfo=None)).total_seconds() / 3600 \
                     if latest_update else float("inf")

    violations = []
    if missing:           violations.append(f"missing_regions: {missing}")
    if total_revenue <= 0: violations.append("non_positive_total_revenue")
    if age_hours > FRESHNESS_HOURS: violations.append(f"stale: {age_hours:.1f}h old")

    return {
        "run_date":       run_date,
        "row_count":      len(rows),
        "total_revenue":  round(total_revenue, 2),
        "regions_found":  list(actual_regions),
        "missing_regions": list(missing),
        "freshness_hours": round(age_hours, 2),
        "passed":         len(violations) == 0,
        "violations":     violations,
    }


gold_read = spark.read.format("delta").load(f"{BASE}/gold/revenue_by_region")
dq        = run_gold_dq(gold_read, RUN_DATE)

print("Gold DQ Report:")
print(json.dumps(dq, indent=2))
info("dq_gold_passed" if dq["passed"] else "dq_gold_failed", **{k: v for k, v in dq.items() if k != "passed"})

if not dq["passed"]:
    raise ValueError(f"Gold DQ failed: {dq['violations']}")

## Part 4: Structured Streaming — Live Order Metrics

The 5-minute real-time dashboard reads from a simulated Kafka stream (rate source locally).

### Interview Cheat Sheet

**Q: Explain the ShopStream streaming architecture end-to-end.**
> Orders are produced to Kafka MSK (orders topic, 8 partitions) by the RDS CDC connector. A long-running EMR cluster runs Spark Structured Streaming with `readStream.format("kafka")`. We parse the JSON payload, apply a 10-minute watermark for late events, then aggregate into 5-minute tumbling windows. Output goes to a Delta streaming table with `trigger(processingTime="5 minutes")`. QuickSight reads this table via Redshift Spectrum → SPICE, refreshing every 5 minutes to meet the SLA. Checkpoint lives on S3 so the job can restart without replaying old data.

**Q: What is a watermark and why do you need it for windowed aggregations?**
> A watermark declares how late an event can arrive and still be included in its window. Without it, Spark would hold the state for every window forever (unbounded memory). With `.withWatermark("event_ts", "10 minutes")`, Spark discards state for windows older than `max_event_time - 10 minutes`. Late events that miss this threshold are dropped. The trade-off: 10-min watermark = 10-min worst-case memory, but also 10-min max lateness tolerance.

**Q: What is `foreachBatch` and when would you use it over a native streaming sink?**
> `foreachBatch(func)` gives you a regular DataFrame for each micro-batch, so you can use any DataFrame API including MERGE INTO, multi-table writes, or custom DQ checks — things the streaming API doesn't support natively. Use it when you need idempotent upserts (Delta MERGE), writing to multiple sinks, or running DQ after each batch. Native sinks (Delta append, Kafka producer) are simpler and preferred when no custom logic is needed.

In [ ]:
# ── Simulate Kafka payload (rate source locally) ───────────────
# In production: .format("kafka").option("subscribe", "orders")
# Locally:       .format("rate") generates rows with [timestamp, value]

REGIONS  = ["US", "EU", "APAC"]
STATUSES = ["shipped", "pending", "cancelled"]

raw_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Simulate order events from the rate source
orders_stream = raw_stream \
    .withColumn("order_id",  F.concat(F.lit("ORD-"), F.lpad(F.col("value").cast("string"), 6, "0"))) \
    .withColumn("user_id",   F.concat(F.lit("USR-"), (F.col("value") % 500).cast("string"))) \
    .withColumn("region",    F.element_at(F.array([F.lit(r) for r in REGIONS]),
                                          (F.col("value") % 3 + 1).cast("int"))) \
    .withColumn("status",    F.element_at(F.array([F.lit(s) for s in STATUSES]),
                                          (F.col("value") % 3 + 1).cast("int"))) \
    .withColumn("amount",    F.round((F.rand() * 1200 + 10), 2)) \
    .withColumn("event_ts",  F.col("timestamp")) \
    .select("order_id", "user_id", "region", "status", "amount", "event_ts")

print("Stream schema:")
orders_stream.printSchema()

In [ ]:
# ── 5-minute tumbling window aggregation ──────────────────────
# Watermark = 10 min: drops late events but bounds state memory

live_metrics = orders_stream \
    .withWatermark("event_ts", "10 minutes") \
    .groupBy(
        F.window("event_ts", "5 minutes").alias("window"),
        "region"
    ) \
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("amount"), 2).alias("total_revenue"),
        F.round(F.avg("amount"), 2).alias("avg_order_value"),
    ) \
    .withColumn("window_start", F.col("window.start")) \
    .withColumn("window_end",   F.col("window.end")) \
    .drop("window")

print("Live metrics schema:")
live_metrics.printSchema()

In [ ]:
# ── Write to Delta streaming table (production pattern) ───────
# outputMode="append": only finalized windows (past watermark) are emitted
# trigger: process every 5 minutes in production, 10s here for demo
# checkpoint: allows restart without replaying from Kafka start

STREAM_PATH      = f"{BASE}/streaming/live_orders"
CHECKPOINT_PATH  = f"{BASE}/checkpoints/live_orders"

def process_batch(batch_df: DataFrame, batch_id: int):
    """foreachBatch handler: validate + write each micro-batch."""
    count = batch_df.count()
    if count == 0:
        return
    info("streaming_batch", batch_id=batch_id, rows=count)
    batch_df.write.format("delta").mode("append").save(STREAM_PATH)


query = live_metrics \
    .writeStream \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime="10 seconds") \
    .foreachBatch(process_batch) \
    .start()

print(f"Streaming query started: id={query.id}, name={query.name}")
print("Waiting 35s for windows to finalize...")
query.awaitTermination(35)
query.stop()
print("Stream stopped.")

In [ ]:
# ── Inspect streaming output ──────────────────────────────────
stream_path = f"{BASE}/streaming/live_orders"

try:
    result = spark.read.format("delta").load(stream_path)
    row_count = result.count()
    if row_count > 0:
        print(f"\nStreaming Delta table: {row_count} rows")
        result.orderBy("window_start", "region").show(20, truncate=False)
    else:
        print("\nNo finalized windows yet — watermark hasn't elapsed.")
        print("(Normal: 5-min windows only close after watermark passes — run longer in prod)")
except Exception:
    print("\nStreaming table not yet written — no finalized windows within the 35s run.")
    print("In production: the 5-min trigger writes completed windows continuously.")

# Show query metrics even if no output rows
print(f"\nQuery status: {query.status}")
print(f"Last progress: {json.dumps(query.lastProgress, indent=2) if query.lastProgress else 'None yet'}")

## Part 5: Production Streaming Patterns (Reference)

The code below shows patterns used in production that require a real Kafka cluster.

In [ ]:
print("""
Production Kafka → Delta streaming (not runnable without MSK):
══════════════════════════════════════════════════════════════

# 1. Read from Kafka
kafka_stream = spark.readStream \\
    .format("kafka") \\
    .option("kafka.bootstrap.servers", "b-1.shopstream.kafka.us-east-1.amazonaws.com:9092") \\
    .option("subscribe", "orders") \\
    .option("startingOffsets", "latest") \\
    .option("kafka.security.protocol", "SASL_SSL") \\
    .option("kafka.sasl.mechanism", "AWS_MSK_IAM") \\
    .option("kafka.sasl.jaas.config", "software.amazon.msk.auth.iam.IAMLoginModule required;") \\
    .load()

# 2. Parse Avro payload (using Schema Registry)
from pyspark.sql.avro.functions import from_avro
schema_str = get_schema_from_registry("orders-value", version="latest")

orders = kafka_stream \\
    .withColumn("payload", from_avro(F.col("value"), schema_str)) \\
    .select("payload.*", F.col("timestamp").alias("kafka_ts"))

# 3. Apply watermark + window
windowed = orders \\
    .withWatermark("event_ts", "10 minutes") \\
    .groupBy(F.window("event_ts", "5 minutes"), "region") \\
    .agg(F.count("*").alias("order_count"), F.sum("amount").alias("total_revenue"))

# 4. foreachBatch with MERGE INTO for idempotent streaming upsert
def upsert_batch(batch_df, batch_id):
    if batch_df.count() == 0: return
    batch_df = batch_df \\
        .withColumn("window_start", F.col("window.start")) \\
        .withColumn("window_end",   F.col("window.end")) \\
        .drop("window")
    
    if DeltaTable.isDeltaTable(spark, STREAM_PATH):
        DeltaTable.forPath(spark, STREAM_PATH).alias("t") \\
            .merge(batch_df.alias("s"),
                   "t.window_start = s.window_start AND t.region = s.region") \\
            .whenMatchedUpdateAll() \\
            .whenNotMatchedInsertAll() \\
            .execute()
    else:
        batch_df.write.format("delta").mode("overwrite").save(STREAM_PATH)

# 5. Start with production trigger
query = windowed.writeStream \\
    .outputMode("append") \\
    .option("checkpointLocation", "s3://shopstream-data/checkpoints/live_orders/") \\
    .trigger(processingTime="5 minutes") \\
    .foreachBatch(upsert_batch) \\
    .start()
══════════════════════════════════════════════════════════════
""")

## Part 6: Full Medallion Summary

Verify all four layers exist and print counts.

In [ ]:
layers = {
    "Bronze (raw)": f"{BASE}/bronze/orders",
    "Silver (clean)": f"{BASE}/silver/orders",
    "Gold/Region": f"{BASE}/gold/revenue_by_region",
    "Gold/Tier": f"{BASE}/gold/revenue_by_tier",
    "Dead-letter": f"{BASE}/dead_letter/orders",
}

print("ShopStream Medallion Architecture — Layer Summary")
print("═" * 52)
for name, path in layers.items():
    try:
        df    = spark.read.format("delta").load(path)
        count = df.count()
        cols  = len(df.columns)
        print(f"  {name:<20} {count:>5} rows  {cols:>3} cols  ✓")
    except Exception as e:
        print(f"  {name:<20}  not yet written")

print("═" * 52)
print()
print("Data flow:  Raw → Bronze → Silver → Gold → BI dashboards")
print("                         ↘ Dead-letter (quarantine)")
print("Kafka → Streaming → Delta streaming table → QuickSight (5-min refresh)")

## Exercises

1. Add a third Gold table: `gold/top_products` that shows the top 10 products by revenue for each `date`. Include `rank` using a window function. Write it idempotently and add a DQ check that verifies exactly 10 rows per date.
2. Modify `build_gold_revenue_by_region()` to also include a `wow_revenue_change_pct` column (week-over-week % change). Since we only have one day of data, simulate this by reading a "prior week" Gold table if it exists; otherwise default to 0.0.
3. The streaming query writes in `append` mode. Explain what happens if the EMR cluster crashes mid-batch and restarts. How does the checkpoint prevent data loss or duplication? Draw the sequence of events.
4. Change the `process_batch` function to: (a) reject batches where avg_order_value < 10 (likely bad data), (b) log a warning instead of writing to Delta, (c) write rejected batches to `{BASE}/streaming/rejected`. Test by temporarily adding a low-amount event.
5. The business asks for a new SLA: Gold must be updated within 1 hour of event time (not 2 hours). Which component changes? Rewrite the Gold DQ `check_freshness` logic to enforce the new SLA and explain what Airflow/EMR changes would be needed.